In [ ]:

import pyodbc
from ollama import chat


# ============================================================
# 1. SQL SERVER CONNECTION
# ============================================================

connection = pyodbc.connect(
    "DRIVER={ODBC Driver 18 for SQL Server};"
    "SERVER=DESKTOP-GC4P189;"
    "DATABASE=orderssystem;"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

cursor = connection.cursor()


# ============================================================
# 2. EXPENSE ANALYSIS AGENT
# ============================================================

def expense_analysis_agent(user_prompt):

    # --------------------------------------------------------
    # QUERY 1: OVERALL EXPENSE SUMMARY
    # --------------------------------------------------------

    cursor.execute("""
        SELECT
            COUNT(*) AS total_expense_entries,
            COALESCE(SUM(amount), 0) AS total_expenses,
            COALESCE(AVG(amount), 0) AS average_expense,
            COALESCE(MIN(amount), 0) AS minimum_expense,
            COALESCE(MAX(amount), 0) AS maximum_expense
        FROM dbo.expense
    """)

    r = cursor.fetchone()

    overall_data = f"""
    Total Expense Entries: {r.total_expense_entries}
    Total Expenses: {r.total_expenses}
    Average Expense: {r.average_expense}
    Minimum Expense: {r.minimum_expense}
    Maximum Expense: {r.maximum_expense}
    """


    # --------------------------------------------------------
    # QUERY 2: CATEGORY-WISE EXPENSES
    # --------------------------------------------------------

    cursor.execute("""
        SELECT
            category,
            COUNT(*) AS expense_entries,
            SUM(amount) AS total_expenses,
            AVG(amount) AS average_expense,
            MIN(amount) AS minimum_expense,
            MAX(amount) AS maximum_expense
        FROM dbo.expense
        GROUP BY category
        ORDER BY total_expenses DESC
    """)

    rows = cursor.fetchall()

    category_data = "\n".join(
        f"Category: {r.category}, "
        f"Entries: {r.expense_entries}, "
        f"Total: {r.total_expenses}, "
        f"Average: {r.average_expense}, "
        f"Minimum: {r.minimum_expense}, "
        f"Maximum: {r.maximum_expense}"
        for r in rows
    ) or "No expense records found."


    # --------------------------------------------------------
    # QUERY 3: MONTHLY EXPENSE ANALYSIS
    # --------------------------------------------------------

    cursor.execute("""
        SELECT
            YEAR(expensedate) AS expense_year,
            MONTH(expensedate) AS expense_month,
            COUNT(*) AS expense_entries,
            SUM(amount) AS total_expenses,
            AVG(amount) AS average_expense
        FROM dbo.expense
        GROUP BY
            YEAR(expensedate),
            MONTH(expensedate)
        ORDER BY
            expense_year,
            expense_month
    """)

    rows = cursor.fetchall()

    monthly_data = "\n".join(
        f"Year: {r.expense_year}, "
        f"Month: {r.expense_month}, "
        f"Entries: {r.expense_entries}, "
        f"Total Expenses: {r.total_expenses}, "
        f"Average Expense: {r.average_expense}"
        for r in rows
    ) or "No monthly expense data found."


    # --------------------------------------------------------
    # QUERY 4: YEARLY EXPENSE ANALYSIS
    # --------------------------------------------------------

    cursor.execute("""
        SELECT
            YEAR(expensedate) AS expense_year,
            COUNT(*) AS expense_entries,
            SUM(amount) AS total_expenses,
            AVG(amount) AS average_expense
        FROM dbo.expense
        GROUP BY YEAR(expensedate)
        ORDER BY expense_year
    """)

    rows = cursor.fetchall()

    yearly_data = "\n".join(
        f"Year: {r.expense_year}, "
        f"Entries: {r.expense_entries}, "
        f"Total Expenses: {r.total_expenses}, "
        f"Average Expense: {r.average_expense}"
        for r in rows
    ) or "No yearly expense data found."


    # --------------------------------------------------------
    # QUERY 5: VENDOR-WISE EXPENSES
    # --------------------------------------------------------

    cursor.execute("""
        SELECT
            COALESCE(vendor, 'Unspecified') AS vendor_name,
            COUNT(*) AS expense_entries,
            SUM(amount) AS total_expenses
        FROM dbo.expense
        GROUP BY COALESCE(vendor, 'Unspecified')
        ORDER BY total_expenses DESC
    """)

    rows = cursor.fetchall()

    vendor_data = "\n".join(
        f"Vendor: {r.vendor_name}, "
        f"Entries: {r.expense_entries}, "
        f"Total Expenses: {r.total_expenses}"
        for r in rows
    ) or "No vendor expense data found."


    # --------------------------------------------------------
    # QUERY 6: LARGEST INDIVIDUAL EXPENSES
    # --------------------------------------------------------

    cursor.execute("""
        SELECT TOP 10
            expenseid,
            expensedate,
            category,
            description,
            amount,
            vendor,
            paymentmethod
        FROM dbo.expense
        ORDER BY amount DESC, expensedate DESC
    """)

    rows = cursor.fetchall()

    largest_expenses = "\n".join(
        f"Expense ID: {r.expenseid}, "
        f"Date: {r.expensedate}, "
        f"Category: {r.category}, "
        f"Description: {r.description}, "
        f"Amount: {r.amount}, "
        f"Vendor: {r.vendor}, "
        f"Payment Method: {r.paymentmethod}"
        for r in rows
    ) or "No expense records found."


    # --------------------------------------------------------
    # QUERY 7: PAYMENT-METHOD ANALYSIS
    # --------------------------------------------------------

    cursor.execute("""
        SELECT
            COALESCE(paymentmethod, 'Unspecified') AS payment_method,
            COUNT(*) AS expense_entries,
            SUM(amount) AS total_expenses
        FROM dbo.expense
        GROUP BY COALESCE(paymentmethod, 'Unspecified')
        ORDER BY total_expenses DESC
    """)

    rows = cursor.fetchall()

    payment_data = "\n".join(
        f"Payment Method: {r.payment_method}, "
        f"Entries: {r.expense_entries}, "
        f"Total Expenses: {r.total_expenses}"
        for r in rows
    ) or "No payment-method data found."


    # --------------------------------------------------------
    # QUERY 8: EXPENSE RECORD QUALITY CHECKS
    # --------------------------------------------------------

    cursor.execute("""
        SELECT
            SUM(
                CASE WHEN category IS NULL
                    OR LTRIM(RTRIM(category)) = ''
                THEN 1 ELSE 0 END
            ) AS missing_categories,

            SUM(
                CASE WHEN description IS NULL
                    OR LTRIM(RTRIM(description)) = ''
                THEN 1 ELSE 0 END
            ) AS missing_descriptions,

            SUM(
                CASE WHEN vendor IS NULL
                    OR LTRIM(RTRIM(vendor)) = ''
                THEN 1 ELSE 0 END
            ) AS missing_vendors

        FROM dbo.expense
    """)

    quality = cursor.fetchone()

    quality_data = f"""
    Records with missing categories: {quality.missing_categories or 0}
    Records with missing descriptions: {quality.missing_descriptions or 0}
    Records with missing vendors: {quality.missing_vendors or 0}
    """


    # --------------------------------------------------------
    # 3. SEND SUMMARIES TO LOCAL OLLAMA MODEL
    # --------------------------------------------------------

    response = chat(
        model="llama3.2",
        messages=[
            {
                "role": "system",
                "content": """
You are an AI Expense Analysis Agent for a business.

Analyse expenses using only the SQL data provided.

RULES:

1. Never invent amounts, dates, categories, or vendors.
2. All amounts are in the currency recorded by the business.
   Do not assume a currency if it has not been specified.
3. Total expenses means the sum of recorded expense entries.
4. Do not confuse expense totals with revenue, product cost,
   gross profit, or net profit.
5. Use actual dates and years from the data.
6. Compare periods only when comparable data is available.
7. For percentage change, use:
   ((Current - Previous) / Previous) * 100.
8. If the previous value is zero, state that percentage
   change cannot be calculated using this formula.
9. Do not claim that an expense is fraudulent or unnecessary
   based only on its amount.
10. Identify unusually high spending as something to investigate,
    not proof of wrongdoing.
11. Explain missing data when it affects the answer.
12. Do not claim that expenses have been reduced or records
    have been changed. This agent only analyses data.
13. Offer practical cost-control suggestions supported by data.
14. If there is insufficient data, state the limitation.
15. No budget data has been supplied unless explicitly included.
    Do not claim that spending is over budget.
16. Provide a clear, structured answer with relevant figures.
"""
            },
            {
                "role": "user",
                "content": f"""
OVERALL EXPENSE SUMMARY:
{overall_data}

CATEGORY-WISE EXPENSES:
{category_data}

MONTHLY EXPENSES:
{monthly_data}

YEARLY EXPENSES:
{yearly_data}

VENDOR-WISE EXPENSES:
{vendor_data}

LARGEST INDIVIDUAL EXPENSES:
{largest_expenses}

PAYMENT-METHOD ANALYSIS:
{payment_data}

EXPENSE DATA QUALITY:
{quality_data}

USER QUESTION:
{user_prompt}

Answer using the supplied information.
Show calculations and comparisons where relevant.
"""
            }
        ]
    )

    return response.message.content


# ============================================================
# 4. USER INTERFACE
# ============================================================

print("\n==============================================")
print("          AI EXPENSE ANALYSIS AGENT")
print("==============================================")

print("\nYou can ask ANY expense-related question.")

print("\nExamples:")
print("  What are my total business expenses?")
print("  Which category has the highest expenses?")
print("  Compare expenses by month.")
print("  Compare yearly expenses.")
print("  What are my five largest expenses?")
print("  Which vendor receives the most payments?")
print("  Compare expenses by payment method.")
print("  Identify possible cost-saving opportunities.")
print("  Check the quality of my expense records.")
print("  Give me a complete expense analysis report.")

print("\nType 'exit' to quit.")


# ============================================================
# 5. ACCEPT ARBITRARY USER PROMPTS
# ============================================================

while True:

    user_prompt = input(
        "\nEnter your expense analysis question: "
    ).strip()

    if user_prompt.lower() == "exit":
        print("\nExiting Expense Analysis Agent...")
        break

    if not user_prompt:
        print("Please enter an expense-related question.")
        continue

    try:
        print("\nAnalysing expense data...")

        result = expense_analysis_agent(user_prompt)

        print("\n==============================================")
        print("              EXPENSE REPORT")
        print("==============================================")
        print(result)
        print("==============================================")

    except Exception as e:
        print("\nERROR:")
        print("----------------------------------------------")
        print(e)


# ============================================================
# 6. CLOSE DATABASE CONNECTION
# ============================================================

cursor.close()
connection.close()

print("\nDatabase connection closed.")
print("Expense Analysis Agent stopped.")



          AI EXPENSE ANALYSIS AGENT

You can ask ANY expense-related question.

Examples:
  What are my total business expenses?
  Which category has the highest expenses?
  Compare expenses by month.
  Compare yearly expenses.
  What are my five largest expenses?
  Which vendor receives the most payments?
  Compare expenses by payment method.
  Identify possible cost-saving opportunities.
  Check the quality of my expense records.
  Give me a complete expense analysis report.

Type 'exit' to quit.



Enter your expense analysis question:  What are my total business expenses?



Analysing expense data...
